# Updates

## Configuration Changes
- **Training Epochs**: Increased from 2 to 3
- **Batch Size**: Reduced from 16 to 8

## Data Processing Improvements
- **Duplicate Removal**: Implemented deduplication logic based on rule and body fields

## Evaluation Enhancements
- **Single Validation Set**: Added evaluation code for standard validation metrics
- **Cross-Validation**: Implemented cross-evaluation framework for robust performance assessment

# 2.Original Notebook
[https://www.kaggle.com/code/daylighth/deberta-small-2epochs-1hr](http://www.kaggle.com/code/daylighth/deberta-small-2epochs-1hr)

In [1]:
%%writefile utils.py
import pandas as pd
import re

def url_to_semantics(text: str) -> str:
    if not isinstance(text, str):
        return ""

    url_pattern = r'https?://[^\s/$.?#].[^\s]*'
    urls = re.findall(url_pattern, text)
    
    if not urls:
        return "" 

    all_semantics = []
    seen_semantics = set()

    for url in urls:
        url_lower = url.lower()
        
        domain_match = re.search(r"(?:https?://)?([a-z0-9\-\.]+)\.[a-z]{2,}", url_lower)
        if domain_match:
            full_domain = domain_match.group(1)
            parts = full_domain.split('.')
            for part in parts:
                if part and part not in seen_semantics and len(part) > 3: # Avoid short parts like 'www'
                    all_semantics.append(f"domain:{part}")
                    seen_semantics.add(part)

        # 2. Extract path parts
        path = re.sub(r"^(?:https?://)?[a-z0-9\.-]+\.[a-z]{2,}/?", "", url_lower)
        path_parts = [p for p in re.split(r'[/_.-]+', path) if p and p.isalnum()] # Split by common delimiters

        for part in path_parts:
            # Clean up potential file extensions or query params
            part_clean = re.sub(r"\.(html?|php|asp|jsp)$|#.*|\?.*", "", part)
            if part_clean and part_clean not in seen_semantics and len(part_clean) > 3:
                all_semantics.append(f"path:{part_clean}")
                seen_semantics.add(part_clean)

    if not all_semantics:
        return ""

    return f"\nURL Keywords: {' '.join(all_semantics)}"


def get_dataframe_to_train(data_path):
    train_dataset = pd.read_csv(f"{data_path}/train.csv") 
    test_dataset = pd.read_csv(f"{data_path}/test.csv")

    flatten = []

    flatten.append(train_dataset[["body", "rule", "subreddit","rule_violation"]].copy())

    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            col_name = f"{violation_type}_example_{i}"
            
            if col_name in train_dataset.columns:
                sub_dataset = train_dataset[[col_name, "rule", "subreddit"]].copy()
                sub_dataset = sub_dataset.rename(columns={col_name: "body"})
                sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0
                
                sub_dataset.dropna(subset=['body'], inplace=True)
                sub_dataset = sub_dataset[sub_dataset['body'].str.strip().str.len() > 0]
                
                if not sub_dataset.empty:
                    flatten.append(sub_dataset)
    
    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            col_name = f"{violation_type}_example_{i}"
            
            if col_name in test_dataset.columns:
                sub_dataset = test_dataset[[col_name, "rule", "subreddit"]].copy()
                sub_dataset = sub_dataset.rename(columns={col_name: "body"})
                sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0
                
                sub_dataset.dropna(subset=['body'], inplace=True)
                sub_dataset = sub_dataset[sub_dataset['body'].str.strip().str.len() > 0]
                
                if not sub_dataset.empty:
                    flatten.append(sub_dataset)
    
    dataframe = pd.concat(flatten, axis=0)
    dataframe = dataframe.drop_duplicates(subset=['body', 'rule', 'subreddit'], ignore_index=True)
    dataframe.drop_duplicates(subset=['body','rule'],keep='first',inplace=True)

     # ================================
    # 🔹 数据增强部分
    # ================================
    # 取30%的数据
    sampled = dataframe.sample(frac=0.3, random_state=42).copy()

    all_rules = dataframe["rule"].unique().tolist()
    def get_different_rule(current_rule):
        choices = [r for r in all_rules if r != current_rule]
        return pd.Series(choices).sample(1).iloc[0]

    sampled["rule"] = sampled["rule"].apply(get_different_rule)
    sampled["rule_violation"] = 0  # 必定为负例

    # 合并增强数据
    dataframe = pd.concat([dataframe, sampled], axis=0).reset_index(drop=True)

    # ✅ 数据增强：随机挑选20%的样本并复制
    # augment_df = dataframe.sample(frac=0.2, random_state=3407)
    # dataframe = pd.concat([dataframe, augment_df], axis=0).reset_index(drop=True)

    dataframe.to_csv("/kaggle/working/dataframe.csv", index=False)
    
    return dataframe.sample(frac=1, random_state=42).reset_index(drop=True)

Writing utils.py


In [2]:
%%writefile train_deberta.py
import os
import pandas as pd
import torch
import random
import numpy as np
from sklearn.model_selection import train_test_split 
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from utils import get_dataframe_to_train, url_to_semantics

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) 
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class CFG:
    model_name_or_path = "/kaggle/input/huggingfacedebertav3variants/deberta-v3-base"
    data_path = "/kaggle/input/jigsaw-agile-community-rules/"
    output_dir = "./deberta_v3_base_final_model"
  
    EPOCHS = 3
    LEARNING_RATE = 2e-5  
    
    MAX_LENGTH = 512
    BATCH_SIZE = 8

class JigsawDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

def main():
    seed_everything(42)
    training_data_df = get_dataframe_to_train(CFG.data_path)
    # training_data_df, valid_df = train_test_split(full_df,test_size=0.2,stratify=full_df['rule'],random_state=42)
    print(f"Training dataset (from examples only) size: {len(training_data_df)}")

    test_df_for_prediction = pd.read_csv(f"{CFG.data_path}/test.csv")
    
    training_data_df['body_with_url'] = training_data_df['body'].apply(lambda x: x + url_to_semantics(x))
    training_data_df['input_text'] = training_data_df['rule'] + "[SEP]" + training_data_df['body_with_url']

    tokenizer = AutoTokenizer.from_pretrained(CFG.model_name_or_path)
    train_encodings = tokenizer(training_data_df['input_text'].tolist(), truncation=True, padding=True, max_length=CFG.MAX_LENGTH)
    train_labels = training_data_df['rule_violation'].tolist()
    train_dataset = JigsawDataset(train_encodings, train_labels)

    model = AutoModelForSequenceClassification.from_pretrained(CFG.model_name_or_path, num_labels=2)
    
    training_args = TrainingArguments(
        output_dir=CFG.output_dir,
        num_train_epochs=CFG.EPOCHS,
        learning_rate=CFG.LEARNING_RATE,
        per_device_train_batch_size=CFG.BATCH_SIZE,
        warmup_ratio=0.1,
        weight_decay=0.01,
        report_to="none",
        save_strategy="no",  #这一行加上这个 save_strategy="no"
        logging_steps=1,
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
    )
    
    trainer.train()

    test_df_for_prediction['body_with_url'] = test_df_for_prediction['body'].apply(lambda x: x + url_to_semantics(x))
    test_df_for_prediction['input_text'] = test_df_for_prediction['rule'] + "[SEP]" + test_df_for_prediction['body_with_url']
    
    test_encodings = tokenizer(test_df_for_prediction['input_text'].tolist(), truncation=True, padding=True, max_length=CFG.MAX_LENGTH)
    test_dataset = JigsawDataset(test_encodings)
    
    predictions = trainer.predict(test_dataset)
    probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=1)[:, 1].numpy()

    submission_df = pd.DataFrame({
        "row_id": test_df_for_prediction["row_id"],
        "rule_violation": probs
    })
    submission_df.to_csv("submission_deberta.csv", index=False)

if __name__ == "__main__":
    main()

Writing train_deberta.py


In [3]:
!python train_deberta.py

2025-10-17 23:25:06.799820: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760743507.197462     187 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760743507.295098     187 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Training dataset (from examples only) size: 2437
/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have conver

In [4]:
! head submission_deberta.csv

row_id,rule_violation
2029,0.016552363
2030,0.021025024
2031,0.51527566
2032,0.72480804
2033,0.86867404
2034,0.01652273
2035,0.8614632
2036,0.015183928
2037,0.016938465


# 4.Qwen3-enmdding

In [5]:
# import os
# import pandas as pd

In [6]:
# %%writefile constants.py
# EMBDEDDING_MODEL_PATH = "/kaggle/input/qwen-3-embedding/transformers/0.6b/1"
# MODEL_OUTPUT_PATH = '/kaggle/input/qwen3-8b-embedding'
# # MODEL_OUTPUT_PATH = '/kaggle/input/qwen3-0-6b-embedding-finetune-epoch2'
# DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules"

# # https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/blob/main/config_sentence_transformers.json
# EMBEDDING_MODEL_QUERY = "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:"

# CLEAN_TEXT = True
# TOP_K = 2000
# BATCH_SIZE = 128

In [7]:
# %%writefile utils.py
# import pandas as pd
# import torch.distributed as dist

# from datasets import Dataset
# from cleantext import clean
# from tqdm.auto import tqdm

# from constants import CLEAN_TEXT


# def build_prompt(row):
#     return f"""r/{row["subreddit"]}\nComment: {row["body"]}"""


# def cleaner(text):
#     return clean(
#         text,
#         fix_unicode=True,
#         to_ascii=True,
#         lower=False,
#         no_line_breaks=False,
#         no_urls=True,
#         no_emails=True,
#         no_phone_numbers=True,
#         no_numbers=False,
#         no_digits=False,
#         no_currency_symbols=False,
#         no_punct=False,
#         replace_with_url="<URL>",
#         replace_with_email="<EMAIL>",
#         replace_with_phone_number="<PHONE>",
#         lang="en",
#     )



# def get_dataframe_to_train(data_path):
#     train_dataset = pd.read_csv(f"{data_path}/train.csv")
#     test_dataset = pd.read_csv(f"{data_path}/test.csv").sample(frac=0.6, random_state=42).reset_index(drop=True)

#     flatten = []
#     flatten.append(train_dataset[["body", "rule", "subreddit", "rule_violation"]])
    
#     for violation_type in ["positive", "negative"]:
#         for i in range(1, 3):
#             sub_dataset = test_dataset[[f"{violation_type}_example_{i}", "rule", "subreddit"]].copy()
#             sub_dataset = sub_dataset.rename(columns={f"{violation_type}_example_{i}": "body"})
#             sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0
#             flatten.append(sub_dataset)

#     dataframe = pd.concat(flatten, axis=0)    
#     dataframe = dataframe.drop_duplicates(ignore_index=True)
#     return dataframe


# def prepare_dataframe(dataframe):
#     dataframe["prompt"] = dataframe.apply(build_prompt, axis=1)

    
#     if CLEAN_TEXT:
#         tqdm.pandas(desc="cleaner")
#         dataframe["prompt"] = dataframe["prompt"].progress_apply(cleaner)

#     if "rule_violation" in dataframe.columns:
#         dataframe["rule_violation"] = dataframe["rule_violation"].map(
#             {
#                 1: 1,
#                 0: -1,
#             }
#         )

#     return dataframe

In [8]:
# %%writefile semantic.py
# import pandas as pd
# from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer
# from sentence_transformers import SentenceTransformer
# from sentence_transformers.util import semantic_search, dot_score
# from tqdm.auto import tqdm
# from peft import PeftModel, PeftConfig


# from utils import get_dataframe_to_train, prepare_dataframe
# from constants import DATA_PATH, EMBDEDDING_MODEL_PATH, EMBEDDING_MODEL_QUERY, TOP_K, BATCH_SIZE, MODEL_OUTPUT_PATH



# def get_scores(test_dataframe):
#     corpus_dataframe = get_dataframe_to_train(DATA_PATH)
#     corpus_dataframe = prepare_dataframe(corpus_dataframe)
    
#     # Load base model
#     model = AutoModelForCausalLM.from_pretrained(EMBDEDDING_MODEL_PATH)
#     tokenizer = AutoTokenizer.from_pretrained(EMBDEDDING_MODEL_PATH)
    
#     # Load adapter configuration and model
#     adapter_config = PeftConfig.from_pretrained(MODEL_OUTPUT_PATH)
#     lora_model = PeftModel.from_pretrained(model, MODEL_OUTPUT_PATH, config=adapter_config)
#     merged_model = lora_model.merge_and_unload()
#     tokenizer.save_pretrained("Qwen3Emb_Finetuned")
#     merged_model.save_pretrained("Qwen3Emb_Finetuned")

#     # 4. Tạo lại SentenceTransformer từ encoder đã merge
#     embedding_model = SentenceTransformer(model_name_or_path="Qwen3Emb_Finetuned", device="cuda")

#     print('Done loading model!')

#     result = []
#     for rule in tqdm(test_dataframe["rule"].unique(), desc=f"Generate scores for each rule"):
#         test_dataframe_part = test_dataframe.query("rule == @rule").reset_index(drop=True)
#         corpus_dataframe_part = corpus_dataframe.query("rule == @rule").reset_index(drop=True)
#         corpus_dataframe_part = corpus_dataframe_part.reset_index(names="row_id")
        
#         query_embeddings = embedding_model.encode(
#             sentences=test_dataframe_part["prompt"].tolist(),
#             prompt=EMBEDDING_MODEL_QUERY,
#             batch_size=BATCH_SIZE,
#             show_progress_bar=True,
#             convert_to_tensor=True,
#             device="cuda",
#             normalize_embeddings=True,
#         )
#         document_embeddings = embedding_model.encode(
#             sentences=corpus_dataframe_part["prompt"].tolist(),
#             batch_size=BATCH_SIZE,
#             show_progress_bar=True,
#             convert_to_tensor=True,
#             device="cuda",
#             normalize_embeddings=True,
#         )
#         test_dataframe_part["semantic"] = semantic_search(
#             query_embeddings,
#             document_embeddings,
#             top_k=TOP_K,
#             score_function=dot_score,
#         )
#         def get_score(semantic):
#             semantic = pd.DataFrame(semantic)
#             semantic = semantic.merge(
#                 corpus_dataframe_part[["row_id", "rule_violation"]],
#                 how="left",
#                 left_on="corpus_id",
#                 right_on="row_id",
#             )
#             semantic["score"] = semantic["score"]*semantic["rule_violation"]
#             return semantic["score"].sum()
            
#         tqdm.pandas(desc=f"Add label for {rule=}")
#         test_dataframe_part["rule_violation"] = test_dataframe_part["semantic"].progress_apply(get_score)
#         result.append(test_dataframe_part[["row_id", "rule_violation"]].copy())
        
#     submission = pd.concat(result, axis=0)
#     return submission


# def generate_submission():
#     test_dataframe = pd.read_csv(f"{DATA_PATH}/test.csv")
#     test_dataframe = prepare_dataframe(test_dataframe)
    
#     submission = get_scores(test_dataframe)
#     submission = test_dataframe[["row_id"]].merge(submission, on="row_id", how="left")
#     submission.to_csv("submission_qwen3.csv", index=False)


# if __name__ == "__main__":
#     generate_submission()



In [9]:
# !python semantic.py

# 5.deberta-2-train

In [10]:
%%writefile utils.py
import pandas as pd
import re

def url_to_semantics(text: str) -> str:
    if not isinstance(text, str):
        return ""

    url_pattern = r'https?://[^\s/$.?#].[^\s]*'
    urls = re.findall(url_pattern, text)
    
    if not urls:
        return "" 

    all_semantics = []
    seen_semantics = set()

    for url in urls:
        url_lower = url.lower()
        
        domain_match = re.search(r"(?:https?://)?([a-z0-9\-\.]+)\.[a-z]{2,}", url_lower)
        if domain_match:
            full_domain = domain_match.group(1)
            parts = full_domain.split('.')
            for part in parts:
                if part and part not in seen_semantics and len(part) > 3: # Avoid short parts like 'www'
                    all_semantics.append(f"domain:{part}")
                    seen_semantics.add(part)

        # 2. Extract path parts
        path = re.sub(r"^(?:https?://)?[a-z0-9\.-]+\.[a-z]{2,}/?", "", url_lower)
        path_parts = [p for p in re.split(r'[/_.-]+', path) if p and p.isalnum()] # Split by common delimiters

        for part in path_parts:
            # Clean up potential file extensions or query params
            part_clean = re.sub(r"\.(html?|php|asp|jsp)$|#.*|\?.*", "", part)
            if part_clean and part_clean not in seen_semantics and len(part_clean) > 3:
                all_semantics.append(f"path:{part_clean}")
                seen_semantics.add(part_clean)

    if not all_semantics:
        return ""

    return f"\nURL Keywords: {' '.join(all_semantics)}"


def get_dataframe_to_train(data_path):
    train_dataset = pd.read_csv(f"{data_path}/train.csv") 
    test_dataset = pd.read_csv(f"{data_path}/test.csv")

    flatten = []

    flatten.append(train_dataset[["body", "rule", "subreddit","rule_violation"]].copy())

    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            col_name = f"{violation_type}_example_{i}"
            
            if col_name in train_dataset.columns:
                sub_dataset = train_dataset[[col_name, "rule", "subreddit"]].copy()
                sub_dataset = sub_dataset.rename(columns={col_name: "body"})
                sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0
                
                sub_dataset.dropna(subset=['body'], inplace=True)
                sub_dataset = sub_dataset[sub_dataset['body'].str.strip().str.len() > 0]
                
                if not sub_dataset.empty:
                    flatten.append(sub_dataset)
    
    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            col_name = f"{violation_type}_example_{i}"
            
            if col_name in test_dataset.columns:
                sub_dataset = test_dataset[[col_name, "rule", "subreddit"]].copy()
                sub_dataset = sub_dataset.rename(columns={col_name: "body"})
                sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0
                
                sub_dataset.dropna(subset=['body'], inplace=True)
                sub_dataset = sub_dataset[sub_dataset['body'].str.strip().str.len() > 0]
                
                if not sub_dataset.empty:
                    flatten.append(sub_dataset)
    
    dataframe = pd.concat(flatten, axis=0)
    dataframe = dataframe.drop_duplicates(subset=['body', 'rule', 'subreddit'], ignore_index=True)
    dataframe.drop_duplicates(subset=['body','rule'],keep='first',inplace=True)

    # ✅ 数据增强：随机挑选20%的样本并复制
    # augment_df = dataframe.sample(frac=0.2, random_state=10086)
    # dataframe = pd.concat([dataframe, augment_df], axis=0).reset_index(drop=True)

     # ================================
    # 🔹 数据增强部分
    # ================================
    # 取30%的数据
    # sampled = dataframe.sample(frac=0.4, random_state=3407).copy()

    # all_rules = dataframe["rule"].unique().tolist()
    # def get_different_rule(current_rule):
    #     choices = [r for r in all_rules if r != current_rule]
    #     return pd.Series(choices).sample(1).iloc[0]

    # sampled["rule"] = sampled["rule"].apply(get_different_rule)
    # sampled["rule_violation"] = 0  # 必定为负例

    # # 合并增强数据
    # dataframe = pd.concat([dataframe, sampled], axis=0).reset_index(drop=True)

    dataframe.to_csv("/kaggle/working/dataframe.csv", index=False)
    
    return dataframe.sample(frac=1, random_state=42).reset_index(drop=True)

Overwriting utils.py


In [11]:
%%writefile train_deberta_small.py
import os
import pandas as pd
import torch
import random
import numpy as np
from sklearn.model_selection import train_test_split 
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from utils import get_dataframe_to_train, url_to_semantics

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) 
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class CFG:
    model_name_or_path = "/kaggle/input/huggingfacedebertav3variants/deberta-v3-small"
    data_path = "/kaggle/input/jigsaw-agile-community-rules/"
    output_dir = "./deberta-v3-small_final_model"
  
    EPOCHS = 3
    LEARNING_RATE = 2e-5  
    
    MAX_LENGTH = 512
    BATCH_SIZE = 8

class JigsawDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

def main():
    seed_everything(42)
    training_data_df = get_dataframe_to_train(CFG.data_path)
    # training_data_df, valid_df = train_test_split(full_df,test_size=0.2,stratify=full_df['rule'],random_state=42)
    print(f"Training dataset (from examples only) size: {len(training_data_df)}")

    test_df_for_prediction = pd.read_csv(f"{CFG.data_path}/test.csv")
    
    training_data_df['body_with_url'] = training_data_df['body'].apply(lambda x: x + url_to_semantics(x))
    training_data_df['input_text'] = training_data_df['rule'] + "[SEP]" + training_data_df['body_with_url']

    tokenizer = AutoTokenizer.from_pretrained(CFG.model_name_or_path)
    train_encodings = tokenizer(training_data_df['input_text'].tolist(), truncation=True, padding=True, max_length=CFG.MAX_LENGTH)
    train_labels = training_data_df['rule_violation'].tolist()
    train_dataset = JigsawDataset(train_encodings, train_labels)

    model = AutoModelForSequenceClassification.from_pretrained(CFG.model_name_or_path, num_labels=2)
    
    training_args = TrainingArguments(
        output_dir=CFG.output_dir,
        num_train_epochs=CFG.EPOCHS,
        learning_rate=CFG.LEARNING_RATE,
        per_device_train_batch_size=CFG.BATCH_SIZE,
        warmup_ratio=0.1,
        weight_decay=0.01,
        report_to="none",
        save_strategy="no",  #这一行加上这个 save_strategy="no"
        logging_steps=1,
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
    )
    
    trainer.train()

    test_df_for_prediction['body_with_url'] = test_df_for_prediction['body'].apply(lambda x: x + url_to_semantics(x))
    test_df_for_prediction['input_text'] = test_df_for_prediction['rule'] + "[SEP]" + test_df_for_prediction['body_with_url']
    
    test_encodings = tokenizer(test_df_for_prediction['input_text'].tolist(), truncation=True, padding=True, max_length=CFG.MAX_LENGTH)
    test_dataset = JigsawDataset(test_encodings)
    
    predictions = trainer.predict(test_dataset)
    probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=1)[:, 1].numpy()

    submission_df = pd.DataFrame({
        "row_id": test_df_for_prediction["row_id"],
        "rule_violation": probs
    })
    submission_df.to_csv("submission_small.csv", index=False)

if __name__ == "__main__":
    main()

Writing train_deberta_small.py


In [12]:
!python train_deberta_small.py

2025-10-17 23:31:28.783333: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760743888.805250    1182 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760743888.811948    1182 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Training dataset (from examples only) size: 1875
/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have conver

In [13]:
!head submission_small.csv

row_id,rule_violation
2029,0.05530482
2030,0.6835703
2031,0.55975807
2032,0.97014946
2033,0.9554168
2034,0.026319366
2035,0.95258886
2036,0.011731888
2037,0.027790222


# Speed-run

In [14]:
%%writefile constants.py
BASE_MODEL_PATH = "/kaggle/input/qwen3-8b-gptq-int4/keras/default/1"#"/kaggle/input/qwen2.5/transformers/0.5b-instruct-gptq-int4/1"
LORA_PATH = "output-qwen8b/"
DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules/"

POSITIVE_ANSWER = "Yes"
NEGATIVE_ANSWER = "No"
COMPLETE_PHRASE = "Answer:"
BASE_PROMPT = '''You are given a comment from reddit and a rule. Your task is to classify whether the comment violates the rule. Only respond Yes/No.'''

Writing constants.py


In [15]:
%%writefile utils.py
import pandas as pd
from datasets import Dataset
from constants import POSITIVE_ANSWER, NEGATIVE_ANSWER, COMPLETE_PHRASE, BASE_PROMPT
import random, numpy as np
from collections import Counter, defaultdict

# 设置随机种子，保证结果可复现
random.seed(5921)
np.random.seed(2131)


def build_prompt(row):
    """
    构造训练/推理用的 prompt
    输入: row - DataFrame 一行
    输出: 包含 BASE_PROMPT、评论内容、规则、Answer 标签的字符串
    """
    return f"""
{BASE_PROMPT}
Comment: {row["body"]}
Rule: {row["rule"]}
{COMPLETE_PHRASE}"""


def _add_if_new(records, used_keys, text, label, rule, subreddit):
    """
    将样本添加到 records 中，保证 (text, rule) 唯一，不重复。
    
    参数:
        records: list, 用于存储样本字典
        used_keys: set, 存储已出现的 (text, rule) 对
        text: str, 评论内容
        label: int, 1=正例, 0=负例
        rule: str, 对应规则
        subreddit: str, 评论所在的 subreddit
    """
    if pd.isna(text) or pd.isna(rule):
        return
    t = str(text).strip()
    r = str(rule).strip()
    if not t or not r:
        return
    key = (t, r)
    if key in used_keys:  # 去重
        return

    # 处理 subreddit 为空的情况
    sub = "" if pd.isna(subreddit) else str(subreddit)

    # 添加到 records
    records.append(
        {
            "body": t,
            "rule_violation": int(label),
            "rule": r,
            "subreddit": sub,
        }
    )
    used_keys.add(key)


def _most_common(lst):
    """
    返回列表中出现次数最多的元素
    """
    if not lst:
        return None
    return Counter(lst).most_common(1)[0][0]


def get_dataframe_to_train(data_path: str) -> pd.DataFrame:
    """
    构造训练用的 DataFrame。
    
    核心逻辑：
    1) 从 train.csv 和 test.csv 中提取正负例及目标文本
    2) 对每条规则 r，从其他规则的正例中抽取硬负样本
    3) 合并到一个 DataFrame 并去重
    
    返回:
        base_df: pd.DataFrame, 列包括 body, rule_violation, rule, subreddit
    """
    df_train = pd.read_csv(f"{data_path}/train.csv")
    df_test = pd.read_csv(f"{data_path}/test.csv")

    records = []
    used_keys = set()  # 保存唯一 (text, rule) 对

    # ---- 1) 基础样本构建 ----
    for x in (df_test, df_train):
        for _, row in x.iterrows():
            rule = row.get("rule")
            subreddit = row.get("subreddit", "")

            # 添加 2 个正例
            _add_if_new(records, used_keys, row.get("positive_example_1"), 1, rule, subreddit)
            _add_if_new(records, used_keys, row.get("positive_example_2"), 1, rule, subreddit)

            # 添加 2 个负例
            _add_if_new(records, used_keys, row.get("negative_example_1"), 0, rule, subreddit)
            _add_if_new(records, used_keys, row.get("negative_example_2"), 0, rule, subreddit)

            # 如果有标注的正文，则也加入
            if "rule_violation" in x.columns and pd.notna(row.get("rule_violation")):
                _add_if_new(
                    records,
                    used_keys,
                    row.get("body"),
                    int(row.get("rule_violation")),
                    rule,
                    subreddit,
                )

    # 如果没有样本，直接返回空 DataFrame
    if not records:
        return pd.DataFrame(columns=["body", "rule_violation", "rule", "subreddit"])

    base_df = pd.DataFrame.from_records(records)

    # ---- 2) 构造硬负样本 (hard negatives) ----
    # 对每条规则，收集所有正例
    rule_to_pos_samples = defaultdict(list)  # rule -> List[(text, subreddit)]
    for _, r in base_df.iterrows():
        if r.get("rule_violation", 0) == 1:
            rule_to_pos_samples[r["rule"]].append((r["body"], r.get("subreddit", "")))

    all_rules = list(rule_to_pos_samples.keys())

    # 生成每条规则对应的“其他规则正例池”
    rule_to_pool_others = {}
    for r in all_rules:
        pool = []
        for r2 in all_rules:
            if r2 == r:
                continue
            pool.extend(rule_to_pos_samples[r2])
        # 去重，只保留第一次出现的 subreddit
        seen_texts = set()
        dedup_pool = []
        for txt, sub in pool:
            if txt not in seen_texts:
                seen_texts.add(txt)
                dedup_pool.append((txt, sub))
        rule_to_pool_others[r] = dedup_pool

    rng = random.Random(361)
    augmented = []

    # 对每条规则 r，随机抽取硬负样本
    for r in all_rules:
        num_pos = len(rule_to_pos_samples[r])
        if num_pos == 0:
            continue

        length_rule = int(num_pos * 0.3)  # 硬负样本数量 = 正例数量
        # length_rule = num_pos
        pool = rule_to_pool_others.get(r, [])
        if not pool:
            continue

        k = min(length_rule, len(pool))
        sampled = rng.sample(pool, k)  # 抽样

        for txt, original_sub in sampled:
            key = (str(txt), str(r))
            if key not in used_keys:
                augmented.append(
                    {
                        "body": str(txt),
                        "rule_violation": 0,  # 标记为负例
                        "rule": str(r),
                        "subreddit": str(original_sub) if original_sub is not None else "",
                    }
                )
                used_keys.add(key)

    # 合并硬负样本
    if augmented:
        base_df = pd.concat([base_df, pd.DataFrame.from_records(augmented)], ignore_index=True)

    # 去掉绝对重复的行
    base_df = base_df.drop_duplicates(ignore_index=True)
    print(f"[INFO] 去重后样本数量: {len(base_df)} 条")

    return base_df


def build_dataset(dataframe: pd.DataFrame) -> Dataset:
    """
    将 DataFrame 转换为 HuggingFace Dataset，添加 prompt 和 completion
    
    参数:
        dataframe: 原始 DataFrame
    返回:
        Dataset 对象，可直接用于训练
    """
    dataframe = dataframe.copy()
    # 添加 prompt 列
    dataframe["prompt"] = dataframe.apply(build_prompt, axis=1)

    columns = ["prompt"]

    # 如果有 rule_violation 列，则映射为 "Yes"/"No"
    if "rule_violation" in dataframe.columns:
        dataframe["completion"] = dataframe["rule_violation"].map({1: POSITIVE_ANSWER, 0: NEGATIVE_ANSWER})
        columns.append("completion")

    # 保留调试用列
    keep_debug_cols = ["body", "rule", "subreddit", "rule_violation"]
    for c in keep_debug_cols:
        if c in dataframe.columns and c not in columns:
            columns.append(c)

    dataframe_out = dataframe[columns]
    dataset = Dataset.from_pandas(dataframe_out, preserve_index=False)

    # 保存 CSV 以便调试
    dataset.to_pandas().to_csv("/kaggle/working/dataset.csv", index=False)

    return dataset


Overwriting utils.py


In [16]:
%%writefile train.py
import pandas as pd

from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
from tqdm.auto import tqdm
from transformers.utils import is_torch_bf16_gpu_available
from utils import build_dataset, get_dataframe_to_train
from constants import DATA_PATH, BASE_MODEL_PATH, LORA_PATH
import os
# is_rerun = os.getenv('KAGGLE_IS_COMPETITION_RERUN')

def main():
    dataframe = get_dataframe_to_train(DATA_PATH)
    train_dataset = build_dataset(dataframe)
    
    lora_config = LoraConfig(
        r=32,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )
    
    # True только в среде пересчёта сабмита
    is_comp_rerun = str(os.getenv("KAGGLE_IS_COMPETITION_RERUN", "")).lower() in {"1","true","yes"}
    
    # хотим 1 шаг при обычном сохранении версии, и полное обучение при сабмите
    fast_mode = not is_comp_rerun
    
    training_args = SFTConfig(
        num_train_epochs=2,
        max_steps=1 if fast_mode else -1,   # <— инвертировали логику
        # max_steps=-1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        optim="paged_adamw_8bit",
        learning_rate=3e-4,
        weight_decay=0.01,
        max_grad_norm=1.0,
        lr_scheduler_type="linear",
        warmup_ratio=0.03,
        bf16=is_torch_bf16_gpu_available(),
        fp16=not is_torch_bf16_gpu_available(),
        dataloader_pin_memory=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        save_strategy="no",
        report_to="none",
        completion_only_loss=True,
        packing=False,
        remove_unused_columns=False,
        seed=3324
    )
        
    trainer = SFTTrainer(
        BASE_MODEL_PATH,
        args=training_args,
        train_dataset=train_dataset,
        peft_config=lora_config,
    )
    
    trainer.train()
    trainer.save_model(LORA_PATH)


if __name__ == "__main__":
    main()

Writing train.py


In [17]:
%%writefile inference.py
import os
os.environ["VLLM_USE_V1"] = "0"

import math
import vllm
import torch
import pandas as pd
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
from vllm.lora.request import LoRARequest
from utils import build_dataset
from constants import BASE_MODEL_PATH, LORA_PATH, DATA_PATH, POSITIVE_ANSWER, NEGATIVE_ANSWER


def main():
    test_dataframe = pd.read_csv(f"{DATA_PATH}/test.csv")

    # 仅保留必要列
    needed_cols = ["row_id", "body", "rule", "subreddit"]
    missing = [c for c in needed_cols if c not in test_dataframe.columns]
    if missing:
        raise ValueError(f"Expected columns missing in test.csv: {missing}")

    test_dataframe = test_dataframe.dropna(subset=["body"]).reset_index(drop=True)
    if len(test_dataframe) == 0:
        raise ValueError("Empty test dataframe after dropping empty bodies.")

    # 检查GPU数量
    n_gpus = torch.cuda.device_count()
    if n_gpus < 2:
        raise RuntimeError(f"Need at least 2 GPUs, but found {n_gpus}.")
    print(f"Detected {n_gpus} CUDA device(s). Using 2 GPUs for tensor parallelism.")

    # 初始化 vLLM，多GPU自动并行
    llm = vllm.LLM(
        BASE_MODEL_PATH,
        quantization="gptq",
        tensor_parallel_size=2,  # ✅ 使用2张GPU
        gpu_memory_utilization=0.98,
        trust_remote_code=True,
        dtype="half",
        enforce_eager=True,
        max_model_len=2048,
        disable_log_stats=True,
        enable_prefix_caching=True,
        enable_lora=True,
        max_lora_rank=64,
    )

    tokenizer = llm.get_tokenizer()
    mclp = MultipleChoiceLogitsProcessor(tokenizer, choices=[POSITIVE_ANSWER, NEGATIVE_ANSWER])

    test_dataset = build_dataset(test_dataframe)
    texts = test_dataset["prompt"]

    print(f"Start generating on {len(texts)} samples...")

    outputs = llm.generate(
        texts,
        vllm.SamplingParams(
            skip_special_tokens=True,
            max_tokens=1,
            logits_processors=[mclp],
            logprobs=2,
        ),
        use_tqdm=True,
        lora_request=LoRARequest("default", 1, LORA_PATH)
    )

    # 收集log_probs
    log_probs = []
    for out in outputs:
        step_lp = out.outputs[0].logprobs[0]
        lp_map = {lp.decoded_token: lp.logprob for lp in step_lp.values()}
        log_probs.append(lp_map)

    preds_df = pd.DataFrame(log_probs)
    if POSITIVE_ANSWER not in preds_df.columns:
        preds_df[POSITIVE_ANSWER] = float("-inf")
    if NEGATIVE_ANSWER not in preds_df.columns:
        preds_df[NEGATIVE_ANSWER] = float("-inf")

    preds_df = preds_df[[POSITIVE_ANSWER, NEGATIVE_ANSWER]]
    preds_df["row_id"] = test_dataframe["row_id"].values

    # 构建submission
    submission = preds_df[["row_id", POSITIVE_ANSWER]].rename(columns={POSITIVE_ANSWER: "rule_violation"})
    rq = submission['rule_violation'].rank(method='average') / (len(submission) + 1)
    submission['rule_violation'] = rq

    out_name = "submission_speedrun.csv"
    submission.to_csv(out_name, index=False)
    print(f"✅ Saved {out_name}")


if __name__ == "__main__":
    main()


Writing inference.py


In [18]:
%%writefile accelerate_config.yaml
compute_environment: LOCAL_MACHINE
debug: false
deepspeed_config:
  gradient_accumulation_steps: 16
  gradient_clipping: 1.0
  train_batch_size: 32
  train_micro_batch_size_per_gpu: 1
  
  zero_stage: 2
  offload_optimizer_device: none
  offload_param_device: none
  zero3_init_flag: false
  
  stage3_gather_16bit_weights_on_model_save: false
  stage3_max_live_parameters: 1e8
  stage3_max_reuse_distance: 1e8
  stage3_prefetch_bucket_size: 5e7
  stage3_param_persistence_threshold: 1e5
  
  zero_allow_untested_optimizer: true
  zero_force_ds_cpu_optimizer: false
  
  fp16:
    enabled: true
    loss_scale: 0
    initial_scale_power: 16
    loss_scale_window: 1000
    hysteresis: 2
    min_loss_scale: 1
  
distributed_type: DEEPSPEED
downcast_bf16: 'no'
dynamo_config:
  dynamo_backend: INDUCTOR
  dynamo_use_fullgraph: false
  dynamo_use_dynamic: false
enable_cpu_affinity: false
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 2
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false

Writing accelerate_config.yaml


In [19]:
!accelerate launch --config_file accelerate_config.yaml train.py

[2025-10-17 23:34:29,396] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2025-10-17 23:34:30,921] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False
2025-10-17 23:34:31.203804: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760744071.226730    1967 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760744071.233713    1967 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W1017 23:34:35.306000 1967 torch/distributed/run.py:766] 
W1017 23:34:35.306000 1967 torch/distributed/run.py:766] *****************************************
W1017 23:34:35.306000 1967 t

In [20]:
!python inference.py

Detected 2 CUDA device(s). Using 2 GPUs for tensor parallelism.
2025-10-17 23:39:47.723886: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760744387.757174    2181 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760744387.765845    2181 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
INFO 10-17 23:39:55 [__init__.py:235] Automatically detected platform cuda.
`torch_dtype` is deprecated! Use `dtype` instead!
WARNING 10-17 23:40:11 [config.py:3443] Casting torch.bfloat16 to torch.float16.
INFO 10-17 23:40:11 [config.py:1604] Using max model len 2048
WARNING 10-17 23:40:12 [config.py:1084] gptq quantization is not fully optimized yet. The speed can be s

# Speed-run2

In [21]:
%%writefile constants.py
BASE_MODEL_PATH = "/kaggle/input/qwen3-1.7b-gptq-int4/keras/default/1"#"/kaggle/input/qwen2.5/transformers/0.5b-instruct-gptq-int4/1"
LORA_PATH = "output-qwen1.7b/"
DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules/"

POSITIVE_ANSWER = "Yes"
NEGATIVE_ANSWER = "No"
COMPLETE_PHRASE = "Answer:"
BASE_PROMPT = '''You are given a comment from reddit and a rule. Your task is to classify whether the comment violates the rule. Only respond Yes/No.'''

Overwriting constants.py


In [22]:
%%writefile utils.py
import pandas as pd
from datasets import Dataset
from constants import POSITIVE_ANSWER, NEGATIVE_ANSWER, COMPLETE_PHRASE, BASE_PROMPT
import random, numpy as np
from collections import Counter, defaultdict

# 设置随机种子，保证结果可复现
random.seed(5921)
np.random.seed(2131)


def build_prompt(row):
    """
    构造训练/推理用的 prompt
    输入: row - DataFrame 一行
    输出: 包含 BASE_PROMPT、评论内容、规则、Answer 标签的字符串
    """
    return f"""
{BASE_PROMPT}
Comment: {row["body"]}
Rule: {row["rule"]}
{COMPLETE_PHRASE}"""


def _add_if_new(records, used_keys, text, label, rule, subreddit):
    """
    将样本添加到 records 中，保证 (text, rule) 唯一，不重复。
    
    参数:
        records: list, 用于存储样本字典
        used_keys: set, 存储已出现的 (text, rule) 对
        text: str, 评论内容
        label: int, 1=正例, 0=负例
        rule: str, 对应规则
        subreddit: str, 评论所在的 subreddit
    """
    if pd.isna(text) or pd.isna(rule):
        return
    t = str(text).strip()
    r = str(rule).strip()
    if not t or not r:
        return
    key = (t, r)
    if key in used_keys:  # 去重
        return

    # 处理 subreddit 为空的情况
    sub = "" if pd.isna(subreddit) else str(subreddit)

    # 添加到 records
    records.append(
        {
            "body": t,
            "rule_violation": int(label),
            "rule": r,
            "subreddit": sub,
        }
    )
    used_keys.add(key)


def _most_common(lst):
    """
    返回列表中出现次数最多的元素
    """
    if not lst:
        return None
    return Counter(lst).most_common(1)[0][0]


def get_dataframe_to_train(data_path: str) -> pd.DataFrame:
    """
    构造训练用的 DataFrame。
    
    核心逻辑：
    1) 从 train.csv 和 test.csv 中提取正负例及目标文本
    2) 对每条规则 r，从其他规则的正例中抽取硬负样本
    3) 合并到一个 DataFrame 并去重
    
    返回:
        base_df: pd.DataFrame, 列包括 body, rule_violation, rule, subreddit
    """
    df_train = pd.read_csv(f"{data_path}/train.csv")
    df_test = pd.read_csv(f"{data_path}/test.csv")

    records = []
    used_keys = set()  # 保存唯一 (text, rule) 对

    # ---- 1) 基础样本构建 ----
    for x in (df_test, df_train):
        for _, row in x.iterrows():
            rule = row.get("rule")
            subreddit = row.get("subreddit", "")

            # 添加 2 个正例
            _add_if_new(records, used_keys, row.get("positive_example_1"), 1, rule, subreddit)
            _add_if_new(records, used_keys, row.get("positive_example_2"), 1, rule, subreddit)

            # 添加 2 个负例
            _add_if_new(records, used_keys, row.get("negative_example_1"), 0, rule, subreddit)
            _add_if_new(records, used_keys, row.get("negative_example_2"), 0, rule, subreddit)

            # 如果有标注的正文，则也加入
            if "rule_violation" in x.columns and pd.notna(row.get("rule_violation")):
                _add_if_new(
                    records,
                    used_keys,
                    row.get("body"),
                    int(row.get("rule_violation")),
                    rule,
                    subreddit,
                )

    # 如果没有样本，直接返回空 DataFrame
    if not records:
        return pd.DataFrame(columns=["body", "rule_violation", "rule", "subreddit"])

    base_df = pd.DataFrame.from_records(records)

    # ---- 2) 构造硬负样本 (hard negatives) ----
    # 对每条规则，收集所有正例
    rule_to_pos_samples = defaultdict(list)  # rule -> List[(text, subreddit)]
    for _, r in base_df.iterrows():
        if r.get("rule_violation", 0) == 1:
            rule_to_pos_samples[r["rule"]].append((r["body"], r.get("subreddit", "")))

    all_rules = list(rule_to_pos_samples.keys())

    # 生成每条规则对应的“其他规则正例池”
    rule_to_pool_others = {}
    for r in all_rules:
        pool = []
        for r2 in all_rules:
            if r2 == r:
                continue
            pool.extend(rule_to_pos_samples[r2])
        # 去重，只保留第一次出现的 subreddit
        seen_texts = set()
        dedup_pool = []
        for txt, sub in pool:
            if txt not in seen_texts:
                seen_texts.add(txt)
                dedup_pool.append((txt, sub))
        rule_to_pool_others[r] = dedup_pool

    rng = random.Random(361)
    augmented = []

    # 对每条规则 r，随机抽取硬负样本
    for r in all_rules:
        num_pos = len(rule_to_pos_samples[r])
        if num_pos == 0:
            continue

        length_rule = int(num_pos * 0.3)  # 硬负样本数量 = 正例数量
        # length_rule = num_pos
        pool = rule_to_pool_others.get(r, [])
        if not pool:
            continue

        k = min(length_rule, len(pool))
        sampled = rng.sample(pool, k)  # 抽样

        for txt, original_sub in sampled:
            key = (str(txt), str(r))
            if key not in used_keys:
                augmented.append(
                    {
                        "body": str(txt),
                        "rule_violation": 0,  # 标记为负例
                        "rule": str(r),
                        "subreddit": str(original_sub) if original_sub is not None else "",
                    }
                )
                used_keys.add(key)

    # 合并硬负样本
    if augmented:
        base_df = pd.concat([base_df, pd.DataFrame.from_records(augmented)], ignore_index=True)

    # 去掉绝对重复的行
    base_df = base_df.drop_duplicates(ignore_index=True)
    print(f"[INFO] 去重后样本数量: {len(base_df)} 条")

    return base_df


def build_dataset(dataframe: pd.DataFrame) -> Dataset:
    """
    将 DataFrame 转换为 HuggingFace Dataset，添加 prompt 和 completion
    
    参数:
        dataframe: 原始 DataFrame
    返回:
        Dataset 对象，可直接用于训练
    """
    dataframe = dataframe.copy()
    # 添加 prompt 列
    dataframe["prompt"] = dataframe.apply(build_prompt, axis=1)

    columns = ["prompt"]

    # 如果有 rule_violation 列，则映射为 "Yes"/"No"
    if "rule_violation" in dataframe.columns:
        dataframe["completion"] = dataframe["rule_violation"].map({1: POSITIVE_ANSWER, 0: NEGATIVE_ANSWER})
        columns.append("completion")

    # 保留调试用列
    keep_debug_cols = ["body", "rule", "subreddit", "rule_violation"]
    for c in keep_debug_cols:
        if c in dataframe.columns and c not in columns:
            columns.append(c)

    dataframe_out = dataframe[columns]
    dataset = Dataset.from_pandas(dataframe_out, preserve_index=False)

    # 保存 CSV 以便调试
    dataset.to_pandas().to_csv("/kaggle/working/dataset.csv", index=False)

    return dataset


Overwriting utils.py


In [23]:
%%writefile train.py
import pandas as pd

from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
from tqdm.auto import tqdm
from transformers.utils import is_torch_bf16_gpu_available
from utils import build_dataset, get_dataframe_to_train
from constants import DATA_PATH, BASE_MODEL_PATH, LORA_PATH
import os
# is_rerun = os.getenv('KAGGLE_IS_COMPETITION_RERUN')

def main():
    dataframe = get_dataframe_to_train(DATA_PATH)
    train_dataset = build_dataset(dataframe)
    
    lora_config = LoraConfig(
        r=32,
        lora_alpha=32,
        lora_dropout=0.1,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )
    
    # True только в среде пересчёта сабмита
    is_comp_rerun = str(os.getenv("KAGGLE_IS_COMPETITION_RERUN", "")).lower() in {"1","true","yes"}
    
    # хотим 1 шаг при обычном сохранении версии, и полное обучение при сабмите
    fast_mode = not is_comp_rerun
    
    training_args = SFTConfig(
        num_train_epochs=2,
        max_steps=1 if fast_mode else -1,   # <— инвертировали логику
        # max_steps=-1,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        optim="paged_adamw_8bit",
        learning_rate=3e-4,
        weight_decay=0.01,
        max_grad_norm=1.0,
        lr_scheduler_type="linear",
        warmup_ratio=0.03,
        bf16=is_torch_bf16_gpu_available(),
        fp16=not is_torch_bf16_gpu_available(),
        dataloader_pin_memory=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        save_strategy="no",
        report_to="none",
        completion_only_loss=True,
        packing=False,
        remove_unused_columns=False,
        seed=3324
    )
        
    trainer = SFTTrainer(
        BASE_MODEL_PATH,
        args=training_args,
        train_dataset=train_dataset,
        peft_config=lora_config,
    )
    
    trainer.train()
    trainer.save_model(LORA_PATH)


if __name__ == "__main__":
    main()

Overwriting train.py


In [24]:
%%writefile inference.py
import os
os.environ["VLLM_USE_V1"] = "0"

import math
import vllm
import torch
import pandas as pd
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
from vllm.lora.request import LoRARequest
from utils import build_dataset
from constants import BASE_MODEL_PATH, LORA_PATH, DATA_PATH, POSITIVE_ANSWER, NEGATIVE_ANSWER


def main():
    test_dataframe = pd.read_csv(f"{DATA_PATH}/test.csv")

    # 仅保留必要列
    needed_cols = ["row_id", "body", "rule", "subreddit"]
    missing = [c for c in needed_cols if c not in test_dataframe.columns]
    if missing:
        raise ValueError(f"Expected columns missing in test.csv: {missing}")

    test_dataframe = test_dataframe.dropna(subset=["body"]).reset_index(drop=True)
    if len(test_dataframe) == 0:
        raise ValueError("Empty test dataframe after dropping empty bodies.")

    # 检查GPU数量
    n_gpus = torch.cuda.device_count()
    if n_gpus < 2:
        raise RuntimeError(f"Need at least 2 GPUs, but found {n_gpus}.")
    print(f"Detected {n_gpus} CUDA device(s). Using 2 GPUs for tensor parallelism.")

    # 初始化 vLLM，多GPU自动并行
    llm = vllm.LLM(
        BASE_MODEL_PATH,
        quantization="gptq",
        tensor_parallel_size=2,  # ✅ 使用2张GPU
        gpu_memory_utilization=0.98,
        trust_remote_code=True,
        dtype="half",
        enforce_eager=True,
        max_model_len=2048,
        disable_log_stats=True,
        enable_prefix_caching=True,
        enable_lora=True,
        max_lora_rank=64,
    )

    tokenizer = llm.get_tokenizer()
    mclp = MultipleChoiceLogitsProcessor(tokenizer, choices=[POSITIVE_ANSWER, NEGATIVE_ANSWER])

    test_dataset = build_dataset(test_dataframe)
    texts = test_dataset["prompt"]

    print(f"Start generating on {len(texts)} samples...")

    outputs = llm.generate(
        texts,
        vllm.SamplingParams(
            skip_special_tokens=True,
            max_tokens=1,
            logits_processors=[mclp],
            logprobs=2,
        ),
        use_tqdm=True,
        lora_request=LoRARequest("default", 1, LORA_PATH)
    )

    # 收集log_probs
    log_probs = []
    for out in outputs:
        step_lp = out.outputs[0].logprobs[0]
        lp_map = {lp.decoded_token: lp.logprob for lp in step_lp.values()}
        log_probs.append(lp_map)

    preds_df = pd.DataFrame(log_probs)
    if POSITIVE_ANSWER not in preds_df.columns:
        preds_df[POSITIVE_ANSWER] = float("-inf")
    if NEGATIVE_ANSWER not in preds_df.columns:
        preds_df[NEGATIVE_ANSWER] = float("-inf")

    preds_df = preds_df[[POSITIVE_ANSWER, NEGATIVE_ANSWER]]
    preds_df["row_id"] = test_dataframe["row_id"].values

    # 构建submission
    submission = preds_df[["row_id", POSITIVE_ANSWER]].rename(columns={POSITIVE_ANSWER: "rule_violation"})
    rq = submission['rule_violation'].rank(method='average') / (len(submission) + 1)
    submission['rule_violation'] = rq

    out_name = "submission_speedrun2.csv"
    submission.to_csv(out_name, index=False)
    print(f"✅ Saved {out_name}")


if __name__ == "__main__":
    main()


Overwriting inference.py


In [25]:
%%writefile accelerate_config.yaml
compute_environment: LOCAL_MACHINE
debug: false
deepspeed_config:
  gradient_accumulation_steps: 4
  gradient_clipping: 1.0
  train_batch_size: 32
  train_micro_batch_size_per_gpu: 4
  
  zero_stage: 2
  offload_optimizer_device: none
  offload_param_device: none
  zero3_init_flag: false
  
  stage3_gather_16bit_weights_on_model_save: false
  stage3_max_live_parameters: 1e8
  stage3_max_reuse_distance: 1e8
  stage3_prefetch_bucket_size: 5e7
  stage3_param_persistence_threshold: 1e5
  
  zero_allow_untested_optimizer: true
  zero_force_ds_cpu_optimizer: false
  
  fp16:
    enabled: true
    loss_scale: 0
    initial_scale_power: 16
    loss_scale_window: 1000
    hysteresis: 2
    min_loss_scale: 1
  
distributed_type: DEEPSPEED
downcast_bf16: 'no'
dynamo_config:
  dynamo_backend: INDUCTOR
  dynamo_use_fullgraph: false
  dynamo_use_dynamic: false
enable_cpu_affinity: false
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 2
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false

Overwriting accelerate_config.yaml


In [26]:
!accelerate launch --config_file accelerate_config.yaml train.py

[2025-10-17 23:42:09,632] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2025-10-17 23:42:11,675] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False
2025-10-17 23:42:11.971925: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760744531.994274    2454 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760744532.001192    2454 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W1017 23:42:16.143000 2454 torch/distributed/run.py:766] 
W1017 23:42:16.143000 2454 torch/distributed/run.py:766] *****************************************
W1017 23:42:16.143000 2454 t

In [27]:
!python inference.py

Detected 2 CUDA device(s). Using 2 GPUs for tensor parallelism.
2025-10-17 23:44:18.803038: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760744658.825016    2657 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760744658.831818    2657 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
INFO 10-17 23:44:22 [__init__.py:235] Automatically detected platform cuda.
`torch_dtype` is deprecated! Use `dtype` instead!
WARNING 10-17 23:44:37 [config.py:3443] Casting torch.bfloat16 to torch.float16.
INFO 10-17 23:44:37 [config.py:1604] Using max model len 2048
WARNING 10-17 23:44:37 [config.py:1084] gptq quantization is not fully optimized yet. The speed can be s

In [28]:
!head submission_speedrun2.csv

row_id,rule_violation
2029,0.5
2030,0.5
2031,0.7272727272727273
2032,0.6363636363636364
2033,0.2727272727272727
2034,0.18181818181818182
2035,0.09090909090909091
2036,0.36363636363636365
2037,0.8181818181818182


# 6.Ensemble

In [29]:
%%writefile ensemble.py
import pandas as pd
import numpy as np

# q = pd.read_csv('submission_qwen0.6b.csv')
l = pd.read_csv('submission_deberta.csv')
# m = pd.read_csv('submission_qwen14b.csv')
# r = pd.read_csv('submission_qwen3.csv')
z = pd.read_csv('submission_small.csv')
a = pd.read_csv('submission_speedrun.csv')
b = pd.read_csv('submission_speedrun2.csv')






# rq = q['rule_violation'].rank(method='average') / (len(q)+1) #0.6b                 3h
rl = l['rule_violation'].rank(method='average') / (len(l)+1) #deberta              1h
# rm = m['rule_violation'].rank(method='average') / (len(m)+1) #14b                 2.5h
# rr = r['rule_violation'].rank(method='average') / (len(r)+1) #qwen3-emdding         0.75h
rz = z['rule_violation'].rank(method='average') / (len(z)+1) #deberta-small         0.5h
ra = a['rule_violation'].rank(method='average') / (len(a)+1) #qwen3-4b-gptq-int4    9h
rb = b['rule_violation'].rank(method='average') / (len(b)+1) #qwen3-1.7b-gptq-int4  2h

blend =  0.45*ra + 0.20*rb + 0.23*rl + 0.12*rz 
# blend =  0.23*rl +  0.17*rz + 0.45*ra + 0.25*rb  # or tune the rank-weights with a tiny grid using OOF
# blend = 0.19*rq + 0.17*rl + 0.18*rr + 0.15*rz + 0.32*ra + 0.22*rb  # or tune the rank-weights with a tiny grid using OOF
# blend = 0.22*rq + 0.17*rl + 0.19*rm + 0.18*rr + 0.15*rz + 0.32*ra  # or tune the rank-weights with a tiny grid using OOF

a['rule_violation'] = blend
a.to_csv('/kaggle/working/submission.csv', index=False)

Writing ensemble.py


In [30]:
!python ensemble.py

In [31]:
import os
import pandas as pd

# 检查 submission.csv 是否存在
if not os.path.exists("submission.csv"):
    print("⚠️ submission.csv 不存在，创建一个空文件...")
    df_empty = pd.DataFrame(columns=["row_id", "rule_violation"])
    df_empty.to_csv("submission.csv", index=False)

# 现在可以安全使用 !head 查看
# !head submission.csv

In [32]:
# 现在可以安全使用 !head 查看
!head submission.csv

row_id,rule_violation
2029,0.37000000000000005
2030,0.3109090909090909
2031,0.6936363636363637
2032,0.7100000000000001
2033,0.5554545454545454
2034,0.22272727272727275
2035,0.5481818181818182
2036,0.18636363636363634
2037,0.5254545454545455
